<p align="center"><img src="https://yolovx.com/wp-content/uploads/2023/11/yolovx_logo.png" width="180" alt="YOLOvX logo"></p>
<h1 align="center" style="text-align: center;">YOLOvX Training Pipeline</h1>

<p align="center"><strong>Runtime Setup:</strong> Runtime → Change runtime type → <strong>T4 GPU</strong> → Save, then Run All.</p>

In [ ]:
!pip install -q fastapi uvicorn python-multipart requests ultralytics pydantic

import os

if not os.path.exists('cloudflared-linux-amd64'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system('chmod +x cloudflared-linux-amd64')

!wget -q -O finetune_worker.py https://raw.githubusercontent.com/wiserli/notebooks/main/finetune_worker.py

## Start Server
Run each session. Copy the generated `trycloudflare.com` URL into your Web App's Training Page.

In [ ]:
import threading, uvicorn, subprocess, time, re, sys, asyncio
sys.path.insert(0, '.')

TUNNEL_URL = None

def start_tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8000'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    for line in iter(p.stderr.readline, b''):
        line = line.decode('utf-8')
        if '.trycloudflare.com' in line:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                TUNNEL_URL = m.group(0)
                print(f'\n{TUNNEL_URL}\n')
                break

def run_server():
    import finetune_worker
    uvicorn.run(finetune_worker.app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=start_tunnel, daemon=True).start()
threading.Thread(target=run_server, daemon=True).start()

try:
    while True:
        await asyncio.sleep(1)
except KeyboardInterrupt:
    pass